# Threads
Move blocking I/O off the event loop.


In [ ]:
# Move blocking I/O to a thread so it does not freeze the event loop.
import asyncio
import time

def blocking_read() -> str:
    time.sleep(0.01)
    return "done"

print(await asyncio.to_thread(blocking_read))


## Polished version
Wrap a blocking library behind an asynchronous interface.


In [ ]:
# Present an async interface even though the third-party library is blocking.
from typing import Protocol

class EmailGateway(Protocol):
    async def send(self, recipient: str, message: str) -> None: ...

class BlockingEmailLibrary:
    def send(self, recipient: str, message: str) -> None:
        time.sleep(0.01)
        print(f"sent to {recipient}: {message}")

class ThreadedEmailGateway:
    def __init__(self, library: BlockingEmailLibrary) -> None:
        self.library = library

    async def send(self, recipient: str, message: str) -> None:
        # Only the blocking call crosses the thread boundary.
        await asyncio.to_thread(self.library.send, recipient, message)

gateway: EmailGateway = ThreadedEmailGateway(BlockingEmailLibrary())
await gateway.send("ada@example.com", "Welcome")


## Applied in this repository

The REST [password adapter](../00P1-project-rest-api/app/infrastructure/security.py) exposes an async interface and runs blocking Argon2 hashing and verification in worker threads.